# Phase 3: Benchmark Results Analysis

This notebook analyzes evaluation results from LMEvalJob runs and provides interactive exploration of benchmark scores across models and datasets.

## What this notebook does

1. **Load results** from completed LMEvalJob runs (JSON format)
2. **Normalize & aggregate** scores by task, category, and supercategory
3. **Compare models** side-by-side with tables and visualizations
4. **Export** results to Markdown or HTML using `generate_report.py`

## Expected Result Format

```
results/
├── gemma-4-E2B-it/
│   ├── kmmlu_direct_law.json
│   ├── click.json
│   └── kobest_wic.json
├── llama-3.1-8b-instruct/
│   └── kmmlu_direct_law.json
└── ...
```

## Step 1: Configuration

In [ ]:
import os
import json
import glob
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

RESULTS_DIR = Path("../results")
NAMESPACE = os.getenv("NAMESPACE", "hyo-project")

print(f"Results directory: {RESULTS_DIR.resolve()}")
print(f"Exists: {RESULTS_DIR.exists()}")

## Step 2: Load Results from Completed Jobs

Results can be loaded from:
- Local JSON files saved during previous runs
- Live LMEvalJob status via `oc` CLI

In [ ]:
def load_results_from_files(results_dir: Path) -> dict:
    """Load all result JSON files organized by model name."""
    all_results = {}

    if not results_dir.exists():
        print(f"Results directory not found: {results_dir}")
        return all_results

    for model_dir in sorted(results_dir.iterdir()):
        if not model_dir.is_dir():
            continue
        model_name = model_dir.name
        all_results[model_name] = {}

        for json_file in sorted(model_dir.glob("*.json")):
            try:
                with open(json_file) as f:
                    data = json.load(f)
                task_name = json_file.stem
                all_results[model_name][task_name] = data
            except (json.JSONDecodeError, IOError) as e:
                print(f"  Skipped {json_file}: {e}")

    print(f"Loaded results for {len(all_results)} model(s)")
    for model, tasks in all_results.items():
        print(f"  {model}: {list(tasks.keys())}")

    return all_results

all_results = load_results_from_files(RESULTS_DIR)

### (Optional) Fetch Results from Live LMEvalJob

In [ ]:
import subprocess

def fetch_lmevaljob_results(job_name: str, namespace: str) -> dict:
    """Fetch results from a completed LMEvalJob via oc CLI."""
    result = subprocess.run(
        ["oc", "get", "lmevaljob", job_name, "-n", namespace,
         "-o", "jsonpath={.status.results}"],
        capture_output=True, text=True
    )
    if result.stdout:
        return json.loads(result.stdout)
    print(f"No results for {job_name}: {result.stderr}")
    return {}

def save_results(data: dict, model_name: str, task_name: str, results_dir: Path):
    """Save fetched results to local JSON file."""
    model_dir = results_dir / model_name
    model_dir.mkdir(parents=True, exist_ok=True)
    output_path = model_dir / f"{task_name}.json"
    with open(output_path, "w") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"Saved: {output_path}")

# Example usage:
# results = fetch_lmevaljob_results("eval-custom-kmmlu", NAMESPACE)
# save_results(results, "gemma-4-E2B-it", "kmmlu_direct_law", RESULTS_DIR)

## Step 3: Parse and Normalize Scores

In [ ]:
def extract_scores(results_data: dict) -> list:
    """
    Extract accuracy scores from lm-evaluation-harness result format.
    Returns list of dicts with task, metric, value.
    """
    scores = []
    results = results_data.get("results", {})

    for task_key, metrics in results.items():
        for metric_name, value in metrics.items():
            if metric_name.startswith("alias"):
                continue
            if isinstance(value, (int, float)):
                scores.append({
                    "task": task_key,
                    "metric": metric_name,
                    "value": value
                })
    return scores


def build_score_table(all_results: dict, metric_filter: str = "acc,none") -> dict:
    """
    Build a comparison table: {task: {model: score}}.
    Default metric is accuracy (acc,none) from lm-eval-harness.
    """
    table = {}

    for model_name, tasks in all_results.items():
        for task_file, data in tasks.items():
            scores = extract_scores(data)
            for s in scores:
                if metric_filter and s["metric"] != metric_filter:
                    continue
                task_key = s["task"]
                if task_key not in table:
                    table[task_key] = {}
                table[task_key][model_name] = round(s["value"] * 100, 2)

    return table

score_table = build_score_table(all_results)
print(f"Tasks found: {list(score_table.keys())}")

## Step 4: Display Results

In [ ]:
try:
    import pandas as pd

    if score_table:
        df = pd.DataFrame(score_table).T
        df.index.name = "Task"
        df = df.sort_index()
        display(df.style.format("{:.2f}").highlight_max(axis=1, color="lightgreen"))
    else:
        print("No results to display. Run evaluations first (Phase 1 or 2).")
except ImportError:
    print("pandas not available. Showing raw dict:")
    for task, models in sorted(score_table.items()):
        print(f"\n{task}:")
        for model, score in sorted(models.items(), key=lambda x: -x[1]):
            print(f"  {model}: {score:.2f}%")

## Step 5: Aggregate by Category / Supercategory

Korean benchmarks have hierarchical category structures:
- **KMMLU**: supercategory (STEM, HUMSS, Applied Science, Other) → category (45 subjects)
- **CLIcK**: supercategory (Culture, Language) → category (11 topics)

In [ ]:
KMMLU_SUPERCATEGORY = {
    "STEM": [
        "biology", "chemical_engineering", "chemistry", "civil_engineering",
        "computer_science", "ecology", "electrical_engineering",
        "information_technology", "materials_engineering", "math",
        "mechanical_engineering"
    ],
    "HUMSS": [
        "accounting", "criminal_law", "economics", "education",
        "korean_history", "law", "management",
        "political_science_and_sociology", "psychology",
        "social_welfare", "taxation"
    ],
    "Applied Science": [
        "aviation_engineering_and_maintenance", "electronics_engineering",
        "energy_management", "environmental_science",
        "gas_technology_and_engineering", "geomatics",
        "industrial_engineer", "machine_design_and_manufacturing",
        "maritime_engineering", "nondestructive_testing",
        "railway_and_automotive_engineering",
        "telecommunications_and_wireless_technology"
    ],
    "Other": [
        "agricultural_sciences", "construction", "fashion",
        "food_processing", "health", "interior_architecture_and_design",
        "marketing", "patent", "public_safety", "real_estate",
        "refrigerating_machinery"
    ]
}

CLICK_SUPERCATEGORY = {
    "Culture": [
        "cul_economy", "cul_geography", "cul_history", "cul_law",
        "cul_politics", "cul_kpop", "cul_society", "cul_tradition"
    ],
    "Language": [
        "lang_function", "lang_grammar", "lang_text"
    ]
}


def aggregate_by_supercategory(score_table: dict, mapping: dict, prefix: str = "") -> dict:
    """Aggregate task-level scores into supercategory averages."""
    aggregated = {}

    for supercategory, categories in mapping.items():
        model_scores = {}
        for cat in categories:
            task_key = f"{prefix}{cat}" if prefix else cat
            if task_key in score_table:
                for model, score in score_table[task_key].items():
                    if model not in model_scores:
                        model_scores[model] = []
                    model_scores[model].append(score)

        if model_scores:
            aggregated[supercategory] = {
                model: round(sum(scores) / len(scores), 2)
                for model, scores in model_scores.items()
            }

    return aggregated

# Example:
# kmmlu_agg = aggregate_by_supercategory(score_table, KMMLU_SUPERCATEGORY, prefix="kmmlu_direct_")
# click_agg = aggregate_by_supercategory(score_table, CLICK_SUPERCATEGORY, prefix="click_")

## Step 6: Export to Markdown

In [ ]:
def score_table_to_markdown(score_table: dict, title: str = "Evaluation Results") -> str:
    """Convert score table to markdown format."""
    if not score_table:
        return "No results available."

    all_models = sorted(set(
        model for tasks in score_table.values() for model in tasks.keys()
    ))

    lines = [f"## {title}", ""]
    header = "| Task | " + " | ".join(all_models) + " |"
    sep = "|---" + "|---" * len(all_models) + "|"
    lines.extend([header, sep])

    for task in sorted(score_table.keys()):
        row = f"| {task} |"
        for model in all_models:
            score = score_table[task].get(model, "-")
            if isinstance(score, float):
                row += f" {score:.2f} |"
            else:
                row += f" {score} |"
        lines.append(row)

    return "\n".join(lines)

md_output = score_table_to_markdown(score_table, "Korean LLM Benchmark Results")
print(md_output)

## Step 7: Generate HTML Report

Use the `generate_report.py` script to produce a standalone HTML report with charts and tables.

In [ ]:
!python generate_report.py --results-dir ../results --output report.html

print("\nReport generated! Open report.html in a browser to view.")

## Step 8: Working with Sample Data

If you don't have real results yet, use the provided sample data to test the analysis pipeline:

In [ ]:
sample_result = {
    "results": {
        "kmmlu_direct_law": {
            "alias": "kmmlu_direct_law",
            "acc,none": 0.542,
            "acc_stderr,none": 0.035
        },
        "kmmlu_direct_biology": {
            "alias": "kmmlu_direct_biology",
            "acc,none": 0.623,
            "acc_stderr,none": 0.028
        },
        "kmmlu_direct_computer_science": {
            "alias": "kmmlu_direct_computer_science",
            "acc,none": 0.712,
            "acc_stderr,none": 0.024
        }
    },
    "config": {
        "model": "local-completions",
        "model_args": "model=gemma-4-E2B-it"
    },
    "n-samples": {
        "kmmlu_direct_law": 200,
        "kmmlu_direct_biology": 200,
        "kmmlu_direct_computer_science": 200
    }
}

# Test the extraction
scores = extract_scores(sample_result)
for s in scores:
    print(f"  {s['task']}: {s['metric']} = {s['value']*100:.2f}%")

## Summary

**This notebook provides:**

- Result loading from local files or live LMEvalJob status
- Score normalization and aggregation by category/supercategory
- Side-by-side model comparison tables
- Markdown and HTML export capabilities

**Companion tools:**

- `generate_report.py` — Standalone HTML report generator with charts
- `samples/sample_result.json` — Example result file for testing

**Full benchmark results:** See [evaluate-llm-on-korean-dataset](https://github.com/hyogrin/evaluate-llm-on-korean-dataset) for comprehensive results across models.